In [ ]:
library(fixest)
library(dplyr)
library(tidyr)
library(ggplot2)
library(lubridate)
library(modelsummary)

panel <- read.csv("../data/monthly_panel.csv")
raw_df <- read.csv("../data/sc_control_data_final_with_agg_score_and_vuln_count.csv")

raw_df <- raw_df %>%
    mutate(year_month = floor_date(as_date(published_at), "month"))

In [ ]:
# Dichotomize variables 

bin_zero_pos <- function(x) {              # adoption: 0 vs any positive
  factor(if_else(x < 0.5, "zero", "positive"),
         levels = c("zero", "positive"))
}

bin_ten_less <- function(x) {              # regression: full marks vs not
  factor(if_else(x > 9.5, "ten", "below_ten"),
         levels = c("below_ten", "ten"))
}


binnings <- list(zero_pos = bin_zero_pos, ten_less = bin_ten_less)

chosen_binning <- c(
  # Aggregate_score            = "continuous",
  Maintained_score           = "continuous",
  Code_Review_score          = "continuous",
  Pinned_Dependencies_score  = "zero_pos",
  Security_Policy_score      = "zero_pos",
  Token_Permissions_score    = "zero_pos",
  DependUpTool_score         = "zero_pos",
  Binary_Artifacts_score     = "ten_less",
  Dangerous_Workflow_score   = "ten_less",
  Contrib_score              = "ten_less"
)

binned_metrics <- chosen_binning[chosen_binning != "continuous"]
continuous_metrics <- names(chosen_binning[chosen_binning == "continuous"])

# 1. Create binned columns in panel
panel_binned <- panel
for (m in names(binned_metrics)) {
  f <- binnings[[binned_metrics[[m]]]]
  panel_binned[[paste0(m, "_bin")]] <- f(panel_binned[[m]])
}

# 2. Build formula: binned versions for the binned metrics,
# raw continuous columns for Maintained_score and Code_Review_score
# Aggregate_score is excluded from this model 
bin_terms  <- paste(paste0(names(binned_metrics), "_bin"), collapse = " + ")
cont_terms <- paste(continuous_metrics, collapse = " + ")

fml_all_binned_str <- paste(
  "Vulnerability_count ~",
  paste(c(cont_terms, bin_terms), collapse = " + "), "+",
  controls,
  "|", fe_part
)
fml_all_binned <- as.formula(fml_all_binned_str)

# 3. Run the model
model_all_binned <- feglm(fml_all_binned, data = panel_binned,
                           family = "poisson", cluster = ~package_name)
summary(model_all_binned)